# Price a Million Tokens, Self Hosted Versus API

Companion notebook for the [Cost per Token](https://codemia.io/courses/llm_inference_and_serving/llm_cost_per_token) lesson on Codemia.

**What you will build**: a unit economics model for serving an LLM. You will derive throughput from batch size with the roofline step time, turn the GPU hourly price into cost per million output tokens, see how utilization and the latency target move that number, find the utilization at which self hosting beats an API price, and price two levers from the course: prompt caching and speculative decoding.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Cost per million from first principles

A GPU costs the same per hour whether it is busy or idle, so

`cost per million = hourly price * GPUs / (tokens per second * utilization * 3600) * 1,000,000`

Check the lesson example: 8 GPUs at 4 dollars, 5,000 tokens per second at full load.

In [ ]:
def cost_per_million(hourly_per_gpu, gpus, tokens_per_s, utilization=1.0):
    return hourly_per_gpu * gpus / (tokens_per_s * utilization * 3600) * 1e6

print("full load:       $%.2f per million" % cost_per_million(4, 8, 5000))
print("50%% utilization: $%.2f per million" % cost_per_million(4, 8, 5000, 0.5))
print("1 GPU, 1,200 tok/s, 40%%: $%.2f per million" % cost_per_million(2.5, 1, 1200, 0.4))

## 2. Throughput comes from the batch, and the batch from the latency target

Using the decode roofline, a step takes the weights plus the batch's KV cache divided by bandwidth, plus overhead. The time per output token a user sees is that step time. A tighter TPOT target caps the batch, which caps throughput and raises cost.

In [ ]:
WEIGHT_BYTES = 16e9; KV_PER_TOKEN = 128 * 1024; BW = 3.35e12; OVERHEAD = 0.002
CACHE_TOKENS = 400_000; CTX = 1750

def step_s(batch):
    return (WEIGHT_BYTES + batch * CTX * KV_PER_TOKEN) / BW + OVERHEAD

batches = np.arange(1, CACHE_TOKENS // CTX + 1)
tpot_ms = 1000 * step_s(batches)
tps = batches / step_s(batches)
cost = cost_per_million(2.5, 1, tps, 0.6)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(tpot_ms, tps); ax[0].set_xlabel("TPOT (ms)"); ax[0].set_ylabel("output tokens per second"); ax[0].grid(alpha=0.3)
ax[0].set_title("Looser latency, bigger batch, more throughput")
ax[1].plot(tpot_ms, cost); ax[1].set_xlabel("TPOT (ms)"); ax[1].set_ylabel("$ per million (60% util)"); ax[1].set_yscale("log"); ax[1].grid(alpha=0.3, which="both")
ax[1].set_title("Cost per token falls as the target loosens")
plt.tight_layout(); plt.show()

for target in [8, 10, 15, 20, 30]:
    b = batches[tpot_ms <= target].max() if (tpot_ms <= target).any() else 0
    print("TPOT <= %2d ms: batch %3d, %5.0f tok/s, $%.3f per million" % (target, b, b / step_s(b), cost_per_million(2.5, 1, b / step_s(b), 0.6)))

## 3. When does self hosting win?

Compare with an API that charges a flat price per million output tokens. Self hosting wins only above a break even utilization, and utilization depends on how bursty your traffic is and how fast you can scale.

In [ ]:
api_price = 0.60
b = batches[tpot_ms <= 20].max(); full_tps = b / step_s(b)
util = np.linspace(0.02, 1, 200)
self_cost = cost_per_million(2.5, 1, full_tps, util) * 1.2      # 20% for hosts and operations
breakeven = util[np.argmax(self_cost <= api_price)]
plt.figure(figsize=(6.5, 3.6))
plt.plot(100 * util, self_cost, label="self hosted, incl. 20% overhead")
plt.axhline(api_price, color="black", ls="--", label="API price")
plt.axvline(100 * breakeven, color="#dd6b20", ls=":", label="break even %.0f%%" % (100 * breakeven))
plt.yscale("log"); plt.xlabel("average utilization (%)"); plt.ylabel("$ per million output tokens")
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()

## 4. Levers: prompt caching and speculative decoding

**Prompt caching** bills a cached prefix at a fraction of the input price on cache hits. **Speculative decoding** produces `E(k) = 1 + a + ... + a^k` tokens per target pass for acceptance rate `a`, at the cost of `k` draft passes, which lowers cost per token when the GPU is latency bound rather than batch bound.

In [ ]:
def cache_saving(requests, prompt, prefix, hit, price, cached_price):
    uncached = requests * prompt * price / 1e6
    cached = requests * (prompt - prefix) * price / 1e6 + requests * prefix * (hit * cached_price + (1 - hit) * price) / 1e6
    return uncached, cached

u, c = cache_saving(1_000_000, 5000, 4000, 0.8, 3.0, 0.30)
print("input bill: $%.0f uncached, $%.0f cached, saving $%.0f" % (u, c, u - c))

def spec_speedup(a, k, draft_cost=0.05):
    return sum(a ** i for i in range(k + 1)) / (1 + k * draft_cost)

ks = range(1, 9)
plt.figure(figsize=(6.5, 3.4))
for a in [0.5, 0.7, 0.8, 0.9]:
    plt.plot(ks, [spec_speedup(a, k) for k in ks], marker="o", label="acceptance %.1f" % a)
plt.xlabel("draft tokens k"); plt.ylabel("tokens per target pass, cost adjusted")
plt.legend(); plt.grid(alpha=0.3); plt.title("Speculation pays most when acceptance is high")
plt.show()

## 5. Exercises

1. Move to a 70B model on 4 GPUs at 2.50 dollars each with 320 KiB per token. What TPOT target makes it cost the same per token as the 8B model at 20 ms?
2. Your traffic peaks at 5 times its nightly trough. With autoscaling that takes 10 minutes, estimate the average utilization you can actually reach and redo the break even.
3. Combine levers: prompt caching on the input side and speculative decoding at acceptance 0.8 on the output side. What is the blended cost per request for 3,000 input and 400 output tokens?